# Identifying good reviews

Let's create a model to identify helpful reviews that people vote as helpful. They could be low rating reviews,

## Load Data

In [27]:
from pathlib import Path
from evalforge.utils import *

dataset_path = Path("data/clothes_review_10k.jsonl")
data = load_jsonl(dataset_path)

print(f"Number of examples: {len(data)}")
print(f"Number of reviews: {sum(len(example['reviews']) for example in data)}")
print("-"*100)
pprint(data[0])

Number of examples: 9999
Number of reviews: 159447
----------------------------------------------------------------------------------------------------
{
    "parent_asin": "5781728791",
    "main_category": "AMAZON FASHION",
    "title": "Women's Crewneck Striped Shirt Loose Colorblock Sweatshirt Pullover Top",
    "description": [],
    "average_rating": 4.0,
    "rating_number": 27,
    "asin": "5781728791",
    "features": [
        "Denim",
        "Hand Wash Only",
        "Imported",
        "Features: Crewneck, Long Sleeve, Stripe printed,Casual and basic Shirts for Women",
        "Casual pullover tops, perfect to pair with jeans, leggings,denim shorts",
        "This womens everyday loose striped shirt is perfect for Daily Wear, Party, School, Vacation, Office, Work, Home, Club, Night Out. Also a great choice as a gift for your wife, girlfriend, mom, daughter or sisters.",
        "Hand wash or Machine wash:Recommended with cold water"
    ],
    "price": "None",
    "images"

In [28]:
item = data[0]
reviews = item["reviews"]

## LLM

In [29]:
import weave
import instructor
from litellm import acompletion

client = instructor.from_litellm(acompletion)

In [30]:
system_prompt = """“”"# Constructing a LLM Judge Benchmark

## The Benchmark
I am trying to build a benchmark for an LLM judge. This benchmark requires positive and negative labels for a given AI-generated output. I have a dataset of Amazon product descriptions and customer reviews which I think I can use. I will construct a benchmark dataset of product descriptions and an associated good / bad ground truth set of labels for the quality of the description text.
I will then pass the product descriptions to a LLM Judge and ask it to rate the descriptions. Then I will compare the Judge’s labels with my ground truth labels in order to understand how aligned.

## Using Reviews As A Proxy signal For Description Text Quality
I want to use the product descriptions as a proxy for AI-generated output and use the feedback provided in the customer review texts as a proxy signal into the quality of the description text.

## Product Description Constraints
However I do not have the actual products in my hand so I am constrained to only assessing the quality of the description text, without knowing if the text matches the actual product in reality. Therefore I am just assessing whether the description text is clear and well-written or whether it is missing information or badly formatted or uses bad english, bad grammar etc. I cannot know if the description text is misleading as I cannot compare to the actual product in reality.

## Scoring Rubric
Given the customer reviews of descriptions please judge the following
- `bad_description`: the product description is missing missing key information or is badly worded, uses poor grammar or is poorly styled or formatted.
- `good_description`: the product description is accurate and helpful.
- `other` the review doesn’t mention the product description or says is misleading.
"""

prompt_template = """The item to review is:

## Item Name
{title}

## Item Description
{description}

## Average Rating
{average_rating}

## Features
{features}

## Review

Rating: {review_rating}
Title: {review_title}

{review_content}
"""


In [31]:
from typing import Literal
from pydantic import BaseModel, Field

class ReviewEvaluation(BaseModel):
    annotation: Literal["bad_description", "good_description", "other"] = Field(description="Is the review helpful and provides useful information about the clothing item?")
    note: str = Field(description="Reason for the evaluation")


In [32]:
def format_example(item: dict, review: dict):
    return prompt_template.format(
        title=item["title"],
        description=listify(item["description"]),
        features=listify(item["features"]),
        average_rating=item["average_rating"],
        review_rating=review["rating"],
        review_title=review["title"],
        review_content=review["text"],
    )

In [33]:
print(format_example(item, reviews[0]))

The item to review is:

## Item Name
Women's Crewneck Striped Shirt Loose Colorblock Sweatshirt Pullover Top

## Item Description
- None

## Average Rating
4.0

## Features
- Denim
- Hand Wash Only
- Imported
- Features: Crewneck, Long Sleeve, Stripe printed,Casual and basic Shirts for Women
- Casual pullover tops, perfect to pair with jeans, leggings,denim shorts
- This womens everyday loose striped shirt is perfect for Daily Wear, Party, School, Vacation, Office, Work, Home, Club, Night Out. Also a great choice as a gift for your wife, girlfriend, mom, daughter or sisters.
- Hand wash or Machine wash:Recommended with cold water

## Review

Rating: 4.0
Title: Soft and light

Bought for a Waldo costume. A little more burgundy than bright red. Material feels a little cheap, it is dyed red so you can see the white through if you stretch it. The inside is a light fleece feeling.



## Helpful reviews

Let's look at the helpful reviews

In [34]:
HELPFUL_VOTE_THRESHOLD = 10

In [43]:
helpful_reviews = []

for item in data:
    for review in item["reviews"]:
        if review["helpful_vote"] > HELPFUL_VOTE_THRESHOLD:
            item_without_reviews = {k: v for k, v in item.items() if k != "reviews"}
            helpful_reviews.append({"item":item_without_reviews, "review": review})

print(f"Number of helpful reviews: {len(helpful_reviews)}")


Number of helpful reviews: 1776


In [44]:
import asyncio
from tqdm.asyncio import tqdm


async def async_map(func, items, max_concurrent=5, desc="Processing"):
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def wrapped_func(**item):
        async with semaphore:
            return await func(**item)
    
    tasks = [wrapped_func(**item) for item in items]
    return await tqdm.gather(*tasks, desc=desc)

@weave.op
async def evaluate_review(item_and_review):
    """Evaluate a single review using the LLM"""
    item, review = item_and_review
    review_evaluation = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, 
                {"role": "user", "content": format_example(item, review)}],
        response_model=ReviewEvaluation,
    )
    return {"review_evaluation": review_evaluation}

## Dataset

In [42]:
import weave

weave.init("amazon_fashion")

annotations = await async_map(evaluate_review, helpful_reviews[:100], max_concurrent=25, desc="Processing reviews")

Logged in as Weights & Biases user: capecape.
View Weave data at https://wandb.ai/capecape/amazon_fashion/weave


Processing reviews:   7%|▋         | 7/100 [00:01<00:13,  7.00it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3bab-7b40-9bed-8974581873b7
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b7e-7a11-846e-812b8783187b
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3ba6-77b1-8e6f-9cc556289e3e
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3baa-71a0-8f01-5992dc11ab95
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b7d-7392-af8f-b03a6dee8b4d
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b84-72d1-921d-759686bf4f5b
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b79-7611-a76d-dd508e0cea43
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b9e-7b83-8983-67cfca8bd19f
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b7f-7f80-b000-53e65f9a5097
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3bac-7410-b92a-d36a44261b65
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b75-7632-8178-3ae4419e7e21
🍩 https://wandb.ai/capecape/amazon_fashion/

Processing reviews:  18%|█▊        | 18/100 [00:01<00:04, 16.81it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3ba8-7492-a5f6-13b33491b56e
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3ba4-7a42-95b3-de03a1040bd1
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3ba7-7c32-a519-760b80d1fe20
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3ba0-70d1-a0e5-907b7af4bfd2
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b65-7500-8130-844dceaf6fa5
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b86-7041-bc3d-c97564d321cd
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-3b7b-7240-9f6e-dab39bbf06c9


...

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-4c86-7ce1-b3ec-2fc3e1dbf5aa
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-4b04-7bf3-8935-59b41b5458f9
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-4e54-7c12-bb29-dc801f7bf106


Processing reviews:  99%|█████████▉| 99/100 [00:06<00:00, 14.00it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-4c2b-7443-840a-edd0035353b2
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-4e46-7e22-980b-60ce5da3e442


Processing reviews: 100%|██████████| 100/100 [00:06<00:00, 15.30it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e324-4cec-7f43-b464-e20addd4f96f
